In [28]:
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer
from torch.optim import AdamW
from tqdm import tqdm
from tokenizers import Tokenizer 
from sklearn.metrics import accuracy_score
import os
import wandb
import toml
from huggingface_hub import HfApi
from sklearn.metrics import classification_report

In [29]:
 # --- Configuration ---
config_file_path = '/Users/nishitha/Desktop/Learn/NLP/Clinical trials eligibility/config.toml'
if not os.path.exists(config_file_path):
    print(f"Error: Config file not found at {config_file_path}")
else:
    config = toml.load(config_file_path)

   # For RNN model parameters
    BATCH_SIZE = config['training']['batch_size']
    NUM_LABELS = config['data']['num_labels']
    LEARNING_RATE = config['training']['learning_rate']
    MAX_LEN_RNN = config['data']['max_len']
    EPOCHS_RNN = config['training']['rnn']['epochs']
    EMBEDDING_DIM = config['model']['embedding_dim']
    RNN_HIDDEN_SIZE = config['model']['rnn']['rnn_hidden_size'] 
    RNN_NUM_LAYERS = config['model']['rnn']['rnn_num_layers']   
    RNN_DROPOUT = config['model']['rnn']['rnn_dropout']        
    DEVICE = torch.device(config['general']['device'])



In [30]:
class DataCreator_RNN(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.patient_texts = list(df['patient'])
        self.criteria_texts = list(df['criteria'])
        self.labels = list(df['label'])
        self.tokenizer = tokenizer
        self.max_len = max_len
        
        self.pad_token_id = self.tokenizer.token_to_id("[PAD]")
        if self.pad_token_id is None:
            print("Warning: [PAD] token not found in tokenizer. Using 0 for padding token ID in DataCreator.")
            self.pad_token_id = 0

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        patient_text = str(self.patient_texts[index])
        criteria_text = str(self.criteria_texts[index])
        label = torch.tensor(self.labels[index], dtype=torch.long)

        encoding = self.tokenizer.encode(criteria_text, patient_text)

        ids = encoding.ids
        attention_mask = encoding.attention_mask

        if len(ids) > self.max_len:
            ids = ids[:self.max_len]
            attention_mask = attention_mask[:self.max_len]
        else:
            padding_length = self.max_len - len(ids)
            ids += [self.pad_token_id] * padding_length
            attention_mask += [0] * padding_length

        return {
            "input_ids": torch.tensor(ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "label": label,
        }


In [33]:
class RNNClassifierFromScratch(nn.Module):
    def __init__(self, vocab_size: int, embedding_dim: int, hidden_size: int, num_layers: int, num_labels: int, dropout_rate: float):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        
        self.lstm = nn.LSTM(embedding_dim, hidden_size, num_layers, 
                            batch_first=True, dropout=dropout_rate if num_layers > 1 else 0)
        
        self.dropout_classifier = nn.Dropout(dropout_rate)
        self.classifier = nn.Linear(hidden_size, num_labels)
        
        self.num_layers = num_layers
        self.hidden_size = hidden_size

    def forward(self, input_ids, attention_mask):
        embedded = self.embedding(input_ids) 

        lengths = attention_mask.sum(dim=1)
        
        lengths = lengths.cpu().clamp(min=1) 

        packed_embedded = nn.utils.rnn.pack_padded_sequence(
            embedded, lengths, batch_first=True, enforce_sorted=False 
        )

        packed_output, (hidden, cell) = self.lstm(packed_embedded)
        
        final_hidden_state = hidden[-1, :, :]
        
        pooled_output = self.dropout_classifier(final_hidden_state)
        
        logits = self.classifier(pooled_output)
        
        return logits



In [35]:
def train_and_evaluate_rnn_model(
    df: pd.DataFrame,
    num_labels: int,
    max_len: int,
    batch_size: int,
    epochs: int,
    learning_rate: float,
    embedding_dim: int,
    hidden_size: int,
    num_rnn_layers: int,
    dropout: float,
    device: torch.device
):
    print(f"\n--- Starting training for Simple RNN Model ---")

    tokenizer = Tokenizer.from_file("/Users/nishitha/Desktop/Learn/NLP/Clinical trials eligibility/BPE/bpe_tokenizer.json")
    vocab_size = tokenizer.get_vocab_size()
    print(f"Custom tokenizer loaded. Vocabulary size: {vocab_size}")

    df['label'] = pd.to_numeric(df['label'], errors='coerce')
    df.dropna(subset=['label'], inplace=True)
    df['label'] = df['label'].astype(int)

    train_val_df, test_df = train_test_split(
        df, test_size=0.2, stratify=df['label'], random_state=42
    )
    train_df, val_df = train_test_split(
        train_val_df, test_size=0.25, stratify=train_val_df['label'], random_state=42
    )

    print(f"Dataset split: Train={len(train_df)} | Val={len(val_df)} | Test={len(test_df)}")

    train_dataset = DataCreator_RNN(df=train_df, tokenizer=tokenizer, max_len=max_len)
    val_dataset = DataCreator_RNN(df=val_df, tokenizer=tokenizer, max_len=max_len)
    test_dataset = DataCreator_RNN(df=test_df, tokenizer=tokenizer, max_len=max_len)

    train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    model = RNNClassifierFromScratch(vocab_size, embedding_dim, hidden_size, num_rnn_layers, num_labels, dropout)
    model.to(device)
    print(f"RNNClassifierFromScratch initialized on {device}.")

    optimizer = AdamW(model.parameters(), lr=learning_rate)
    loss_fn = nn.CrossEntropyLoss()

    best_val_loss = float('inf') 
    patience = 3
    epochs_no_improve = 0

    best_val_accuracy = 0.0
    model_save_path = "scratch_rnn_clinical_model.pt"

    wandb.init(project='NLP_Project_Clinical_Trials', config=config)

    for epoch in range(epochs):
        model.train()
        total_train_loss = 0

        print(f"\nEpoch {epoch + 1}/{epochs} (Model: Simple RNN)")
        loop = tqdm(train_dataloader, leave=True)

        for batch in loop:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask)
            loss = loss_fn(outputs, labels)

            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()
            loop.set_description(f"Epoch {epoch + 1}")
            loop.set_postfix(loss=loss.item())

        avg_train_loss = total_train_loss / len(train_dataloader)
        print(f"Average Training Loss: {avg_train_loss:.4f}")

        model.eval()
        total_val_loss = 0
        correct_val_preds = 0

        with torch.no_grad():
            for batch in val_dataloader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["label"].to(device)

                outputs = model(input_ids, attention_mask)
                loss = loss_fn(outputs, labels)
                total_val_loss += loss.item()
                preds = torch.argmax(outputs, dim=1)
                correct_val_preds += (preds == labels).sum().item()

        avg_val_loss = total_val_loss / len(val_dataloader)
        val_accuracy = correct_val_preds / len(val_dataset)*100

        print(f"Validation loss: {avg_val_loss:.4f}, Accuracy: {val_accuracy:.4f}")

        wandb.log(data={
            "epoch": epoch+1,
            "train_loss": avg_train_loss,
            "val_loss": avg_val_loss,
            "val_accuracy": val_accuracy
        })


        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), model_save_path)
            print(f"Model saved to {model_save_path} (Best validation loss: {best_val_loss:.4f})")
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
        
        if epochs_no_improve == patience:
            print(f"\nEarly stopping triggered after {patience} epochs with no improvement in validation loss.")
            break
    
    print(f"\n--- Finished training for Simple RNN Model ---")

    print(f"\n--- Evaluating Simple RNN Model on the TEST SET ---")
    model.eval()
    total_test_loss = 0
    correct_test_preds = 0
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for batch in tqdm(test_dataloader, desc="Test Evaluation"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            outputs = model(input_ids, attention_mask)
            loss = loss_fn(outputs, labels)
            total_test_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)
            correct_test_preds += (preds == labels).sum().item()
            
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    avg_test_loss = total_test_loss / len(test_dataloader)
    test_accuracy = correct_test_preds / len(test_dataset)

    print("Classification Report for Test Performance")
    print(classification_report(y_true=all_labels, y_pred=all_preds, labels=[0, 1, 2],
                                target_names=["Negative", "Neutral", "Positive"]))

    test_report = classification_report(y_true=all_labels, y_pred=all_preds, labels=[0, 1, 2],
                                target_names=["Negative", "Neutral", "Positive"], output_dict=True)
    wandb.log({"rnn_test_classification_report": test_report})

    print(f"Test Loss: {avg_test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}")


    return {
        "model_name": "Simple RNN Classifier",
        "final_val_loss": avg_val_loss,
        "final_val_accuracy": val_accuracy,
        "best_val_accuracy": best_val_accuracy,
        "final_test_loss": avg_test_loss,
        "final_test_accuracy": test_accuracy,
        "saved_model_path": model_save_path
    }



In [36]:
if __name__ == "__main__":
    df_full = pd.read_csv('../Dataset/cleaned_data_3.csv')

    all_results = []


    model_name_to_test = "dmis-lab/biobert-v1.1"
   
    rnn_result = train_and_evaluate_rnn_model(
        df=df_full.copy(),
        num_labels=NUM_LABELS,
        max_len=MAX_LEN_RNN,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS_RNN,
        learning_rate=LEARNING_RATE,
        embedding_dim=EMBEDDING_DIM,
        hidden_size=RNN_HIDDEN_SIZE,
        num_rnn_layers=RNN_NUM_LAYERS,
        dropout=RNN_DROPOUT,
        device=DEVICE
    )
    all_results.append(rnn_result)
    print(f"RNN Model Results: {rnn_result}")



--- Starting training for Transformer model: dmis-lab/biobert-v1.1 ---
Tokenizer for dmis-lab/biobert-v1.1 loaded.
Dataset split: Train=619 | Val=207 | Test=207
Model dmis-lab/biobert-v1.1 initialized


epoch,▁▁▂▂▃▃▄▄▅▅▆▆▇▇██
train_loss,█▇▅▄▄▃▂▂▁▁▁▁▁▁▁▁
val_accuracy,▁▃▄▄▅▆▇█████████
val_loss,█▆▃▅▄▃▂▁▁▁▁▁▁▁▁▁
epoch,16
train_loss,0.0399
val_accuracy,98.55072
val_loss,0.06226



Epoch 1/25 (Model: dmis-lab/biobert-v1.1)


  0%|          | 0/39 [00:00<?, ?it/s]/Users/nishitha/.local/share/virtualenvs/Clinical_trials_eligibility-lIvJhYYY/lib/python3.9/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
Epoch 1:   3%|▎         | 1/39 [02:40<1:41:29, 160.25s/it, loss=1.14]wandb-core(67474) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
wandb-core(67480) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
wandb-core(67485) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
wandb-core(67489) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
wandb-core(67495) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Epoch 1:   5%|▌         | 2/39 [04:43<1:25:30, 138.66s/it, loss=nan] wandb-c

Average Training Loss: nan


...

wandb-core(71936) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
wandb-core(71941) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
wandb-core(71970) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
wandb-core(71989) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
wandb-core(71992) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Validation loss: nan, Accuracy: 27.5362
Early stopping counter: 3 of 3
Early stopping triggered. Training stopped at epoch 3

--- Finished training for dmis-lab/biobert-v1.1 ---

--- Evaluating dmis-lab/biobert-v1.1 on the TEST SET ---


Test Evaluation: 100%|██████████| 13/13 [01:17<00:00,  5.94s/it]


Test Loss: nan, Test Accuracy: 0.2754

--- Final Comparative Study Summary ---
Model: dmis-lab/biobert-v1.1
  Best Val Accuracy: 27.5362
  Final Test Accuracy: 0.2754
  Final Test Loss: nan
  Saved Model: dmis_lab_biobert_v1.1_clinical_model.pt
------------------------------


wandb-core(72119) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
wandb-core(72175) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
wandb-core(72229) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
wandb-core(72231) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
wandb-core(72232) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
wandb-core(72234) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
wandb-core(72242) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
wandb-core(72243) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
wandb-core(72244) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
wandb-core(72245) MallocStackLogging: can't turn off malloc stack logging because 